# 01 · Carga de datos Excel → Supabase

**Manufactura textil moda Colombia · 89 tiendas · 3 años**

Este notebook lee los archivos Excel de `datos_excel/` y los carga a Supabase.

### Antes de correr
- `.env` en la raíz del proyecto con credenciales
- Tablas creadas en Supabase (`sql/01_crear_tablas.sql`)
- Archivos Excel en `datos_excel/`

In [ ]:
import subprocess, sys
pkgs = ['pandas','openpyxl','sqlalchemy','psycopg2-binary','python-dotenv']
subprocess.run([sys.executable,'-m','pip','install','--quiet']+pkgs)
print('✅ Dependencias listas')

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_engine, get_datos_dir
from sqlalchemy import text

engine    = get_engine()
DATOS_DIR = get_datos_dir()
print(f'📁 {DATOS_DIR}')

## 1 · Dimensiones

In [ ]:
df_tiendas = pd.read_excel(DATOS_DIR / 'dim_tiendas.xlsx')
df_tipos   = pd.read_excel(DATOS_DIR / 'dim_tipos_producto.xlsx')
df_eventos = pd.read_excel(DATOS_DIR / 'dim_eventos.xlsx')
df_eventos['fecha'] = pd.to_datetime(df_eventos['fecha'])
print(f'Tiendas: {len(df_tiendas)} | Tipos: {len(df_tipos)} | Eventos: {len(df_eventos)}')
assert df_tiendas['tienda_id'].nunique()==len(df_tiendas)
assert df_tipos['tipo_producto'].nunique()==len(df_tipos)
print('✅ Validaciones OK')

In [ ]:
with engine.begin() as conn:
    conn.execute(text('DELETE FROM dim_eventos'))
    conn.execute(text('DELETE FROM dim_tipos_producto'))
    conn.execute(text('DELETE FROM dim_tiendas'))
print('🗑️  Dimensiones limpiadas')

df_tiendas.drop(columns=['tienda_id'],errors='ignore').to_sql('dim_tiendas',engine,if_exists='append',index=False,method='multi',chunksize=500)
df_tipos.to_sql('dim_tipos_producto',engine,if_exists='append',index=False,method='multi',chunksize=500)
df_eventos.drop(columns=['evento_id'],errors='ignore').to_sql('dim_eventos',engine,if_exists='append',index=False,method='multi',chunksize=500)
print('✅ dim_tiendas, dim_tipos_producto, dim_eventos cargadas')

## 2 · Ventas 2022–2024

In [ ]:
def limpiar_ventas(df):
    df = df.copy()
    df['fecha'] = pd.to_datetime(df['fecha'])
    devol = (df['unidades_vendidas'] < 0).sum()
    if devol: print(f'  ⚠️  {devol} devoluciones excluidas')
    df = df[df['unidades_vendidas'] > 0]
    df = df.dropna(subset=['fecha','tienda_id','tipo_producto','unidades_vendidas','valor_venta'])
    if 'es_precio_pleno' not in df.columns:
        df['es_precio_pleno'] = df['descuento_pct'] == 0
    return df

with engine.begin() as conn:
    conn.execute(text('DELETE FROM fact_ventas_diarias'))
print('🗑️  fact_ventas_diarias limpiada')

for anio in [2022, 2023, 2024]:
    df_v = limpiar_ventas(pd.read_excel(DATOS_DIR / f'ventas_{anio}.xlsx'))
    df_v.drop(columns=['venta_id'],errors='ignore').to_sql('fact_ventas_diarias',engine,if_exists='append',index=False,method='multi',chunksize=1000)
    print(f'✅ ventas_{anio}: {len(df_v):,} filas | ${df_v["valor_venta"].sum():,.0f} COP | {df_v["es_precio_pleno"].mean()*100:.1f}% precio pleno')

## 3 · Inventario semanal

In [ ]:
with engine.begin() as conn:
    conn.execute(text('DELETE FROM fact_inventario_semanal'))
print('🗑️  fact_inventario_semanal limpiada')

for parte in [1, 2]:
    df_i = pd.read_excel(DATOS_DIR / f'inventario_semanal_parte{parte}.xlsx')
    df_i['fecha'] = pd.to_datetime(df_i['fecha'])
    df_i.drop(columns=['inv_id'],errors='ignore').to_sql('fact_inventario_semanal',engine,if_exists='append',index=False,method='multi',chunksize=1000)
    print(f'✅ inventario parte {parte}: {len(df_i):,} filas')

## 4 · Verificación final

In [ ]:
print('='*50)
print('RESUMEN — SUPABASE')
print('='*50)
with engine.connect() as conn:
    for tabla in ['dim_tiendas','dim_tipos_producto','dim_eventos','fact_ventas_diarias','fact_inventario_semanal']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tabla}')).fetchone()[0]
        print(f'  {tabla:<35} {n:>10,}')
    r = conn.execute(text('SELECT MIN(fecha)::text, MAX(fecha)::text FROM fact_ventas_diarias')).fetchone()
    print(f'\n  Rango ventas: {r[0]} → {r[1]}')
print('='*50)
print('✅ Listo para notebook 02')